In [ ]:
from dotenv import load_dotenv
import requests
import os
from xml.etree import ElementTree as ET
from bs4 import BeautifulSoup
import json


load_dotenv(dotenv_path=r"D:\Shikha\bias_project\.env")
API_KEY = os.getenv("GOVINFO_API_KEY")

# Verify the key loaded
if API_KEY:
    print(f"API key loaded (length: {len(API_KEY)} chars)")
else:
    print("⚠️ ERROR: API key not found. Check .env file exists and has GOVINFO_API_KEY=...")
# Test endpoint: list available collections
url = "https://api.govinfo.gov/collections"
params = {"api_key": API_KEY}

response = requests.get(url, params=params, timeout=30)

print(f"Status code: {response.status_code}")
print(f"Response size: {len(response.text)} chars")

if response.status_code == 200:
    data = response.json()
    print(f"\nNumber of collections available: {len(data.get('collections', []))}")
    print("\nFirst few collections:")
    for coll in data.get('collections', [])[:10]:
        print(f"  - {coll.get('collectionCode'):8s}  {coll.get('collectionName')}")
else:
    print(f"\nError response: {response.text[:500]}")

In [ ]:
# Look specifically for Congressional Record collections
target_codes = ['CREC', 'CRECB', 'CRI']  # daily, bound, index

print("Searching for Congressional Record collections:\n")

found_codes = set()
for coll in data.get('collections', []):
    code = coll.get('collectionCode')
    name = coll.get('collectionName')
    if code in target_codes:
        found_codes.add(code)
        print(f"  ✓ {code:8s}  {name}")

# Report what's missing
missing = set(target_codes) - found_codes
if missing:
    print(f"\nNot found: {missing}")

# Also: full list of what IS available
print(f"\nAll {len(data.get('collections', []))} collections available:")
for coll in data.get('collections', []):
    print(f"  - {coll.get('collectionCode'):10s}  {coll.get('collectionName')}")

In [ ]:
# Pull metadata for one day of Congressional Record
# We'll use a known active session day: January 5, 2023 (118th Congress, Day 1)
# target_date = "2023-03-15"
# target_date = "2023-02-15" 
# target_date = "2023-01-05"
# target_date = "2022-06-15"
target_date = "2024-03-12" 

url = f"https://api.govinfo.gov/published/{target_date}/{target_date}"
params = {
    "api_key": API_KEY,
    "collection": "CREC",
    "pageSize": 10,  # limit results for the demo
    "offsetMark": "*"
}

response = requests.get(url, params=params, timeout=30)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    
    print(f"Total packages available for {target_date}: {data.get('count', 'unknown')}")
    print(f"\nPackages returned (showing first {len(data.get('packages', []))}):\n")
    
    for pkg in data.get('packages', []):
        print(f"  Package ID:    {pkg.get('packageId')}")
        print(f"  Title:         {pkg.get('title')}")
        print(f"  Date:          {pkg.get('dateIssued')}")
        print(f"  Last modified: {pkg.get('lastModified')}")
        print(f"  Package link:  {pkg.get('packageLink')}")
        print()
else:
    print(f"Error: {response.text[:500]}")

In [ ]:
# Try without the collection filter — just see what was published on a given date
target_date = "2024-03-12"

url = f"https://api.govinfo.gov/published/{target_date}/{target_date}"
params = {
    "api_key": API_KEY,
    "pageSize": 10,
    "offsetMark": "*"
}

response = requests.get(url, params=params, timeout=30)
print(f"Status: {response.status_code}")
print(f"URL: {response.url}\n")

if response.status_code == 200:
    data = response.json()
    print(f"Total count: {data.get('count')}")
    print(f"Packages returned: {len(data.get('packages', []))}\n")
    
    # Print the raw response structure to see what's happening
    print("Full response keys:", list(data.keys()))
    print("\nFirst 1000 chars of response:")
    print(response.text[:1000])
else:
    print(f"Error response: {response.text[:500]}")

In [ ]:
import requests



# Use the collections endpoint instead of published
url = "https://api.govinfo.gov/collections/CREC/2024-03-12T00%3A00%3A00Z/2024-03-13T00%3A00%3A00Z"
params = {
    "api_key": API_KEY,
    "pageSize": 10,
    "offsetMark": "*"
}

response = requests.get(url, params=params, timeout=30)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(f"Count: {data.get('count')}")
    print(f"Packages: {len(data.get('packages', []))}")
    for pkg in data.get('packages', [])[:5]:
        print(f"  - {pkg.get('packageId')}: {pkg.get('title')}")
else:
    print(f"Error: {response.text[:300]}")

In [ ]:
import requests


# Try a wider date range and different format
# CREC publishes with a delay, so look at older dates with more buffer
url = "https://api.govinfo.gov/collections/CREC/2024-01-01T00:00:00Z/2024-01-31T23:59:59Z"
params = {
    "api_key": API_KEY,
    "pageSize": 10,
    "offsetMark": "*"
}

response = requests.get(url, params=params, timeout=30)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(f"Count: {data.get('count')}")
    print(f"Packages returned: {len(data.get('packages', []))}\n")
    
    for pkg in data.get('packages', [])[:5]:
        print(f"  Package: {pkg.get('packageId')}")
        print(f"  Title:   {pkg.get('title')}")
        print(f"  Date:    {pkg.get('dateIssued')}")
        print()
else:
    print(f"Error: {response.text[:300]}")

In [ ]:
import requests

# Bulk data URL pattern for a single day's Congressional Record
# Format: https://www.govinfo.gov/bulkdata/CREC/{year}/{month}/CREC-{date}.zip

test_date = "2024-03-12"
year = "2024"
month = "03"

url = f"https://www.govinfo.gov/bulkdata/CREC/{year}/{month}/CREC-{test_date}.zip"
print(f"Fetching: {url}\n")

response = requests.get(url, timeout=60)
print(f"Status: {response.status_code}")
print(f"Content type: {response.headers.get('Content-Type')}")
print(f"Content length: {len(response.content)} bytes")

if response.status_code == 200:
    # Check if it actually looks like a ZIP file
    if response.content[:2] == b'PK':
        print("\n✓ Valid ZIP file received")
    else:
        print(f"\n⚠️ Doesn't look like a ZIP. First 200 chars:")
        print(response.content[:200])
else:
    print(f"\nError response: {response.text[:300]}")

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path

# Check if .env file exists
env_path = Path(r"D:\Shikha\bias_project\.env")
print(f".env file exists: {env_path.exists()}")
print(f".env file size: {env_path.stat().st_size if env_path.exists() else 'N/A'} bytes")

# Load it
load_dotenv(dotenv_path=str(env_path))
API_KEY = os.getenv("GOVINFO_API_KEY")

# Don't print the key — just check it loaded
if API_KEY:
    print(f"\n✓ API_KEY loaded successfully")
    print(f"  Length: {len(API_KEY)} chars")
    print(f"  Starts with: {API_KEY[:4]}***")  # only first 4 chars
else:
    print(f"\n✗ API_KEY is None — .env not loaded properly")